In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib

In [2]:
df_audit_grade_all = pd.read_excel(r'I:\My Drive\Projects_mdw\BayesAudit\Data\Continued shop audit exposure - YTD Oct 2025.xlsx',sheet_name='Oct 25')
print('df_audit_grade',df_audit_grade_all.shape)

df_audit_grade (413, 25)


In [3]:
def preprocess_df_audit_grade(df):
    filter_df = df[['Shop Code','Shop Grade','Current Audit Date']].copy()
    filter_df['Current Audit Date'] = pd.to_datetime(filter_df['Current Audit Date'])
    
    print('df_audit_grade shape :',filter_df.shape)
    print('df_audit_grade Min date :',filter_df['Current Audit Date'].min())
    print('df_audit_grade Max date :',filter_df['Current Audit Date'].max())

    return filter_df

df_audit_grade = preprocess_df_audit_grade(df_audit_grade_all)

df_audit_grade shape : (413, 3)
df_audit_grade Min date : 2024-08-16 00:00:00
df_audit_grade Max date : 2025-10-24 00:00:00


In [4]:
df_audit_grade.head()

,Shop Code,Shop Grade,Current Audit Date
0,BAA01,B,2025-09-11
1,BAB01,C-,2025-08-11
2,BAC01,B,2025-05-14
3,BAD01,A,2025-07-09
4,BAE01,B,2025-02-06


In [5]:
df_kpi_scores = pd.read_csv(r'I:\My Drive\Projects_mdw\BayesAudit\Data\Factors_Values_history.csv')
print(df_kpi_scores.shape)
df_kpi_scores.head()

(58013, 24)


,SITE_ID,EDATE,INVENTORY_TURNOVER_RTO_RISK_SCORE,PRODUCTRELEASE_COST_RTO_RISK_SCORE,BLACKLIST_CUST_ACTIVE_ACC_RISK_SCORE,PH_NO_CHANGE_CUST_RISK_SCORE,EARL_CLOS_DISCOUNT_RISK_SCORE,HIRE_SALE_ASS_ACC_RISK_SCORE,CASHA_RISK_SCORE,ACC_LESS_ADD_PAY_RISK_SCORE,...,TOT_UNPAID_CASH_SALES_RISK,TOT_HP_ARREARS_RISK_RTO,TOT_CRDT_OUTSTN_AMT_DUE_DAYS_RLN,RETURN_CHEQUE_RISK_SCORE,SPECIAL_DISCOUNT_RISK_SCORE,MULTI_HP_ACC_PER_CUST_RISK_SCORE,INITIAL_PAYMENT_NOT_PAID_RISK_SCORE,SALES_RETURN_RISK_SCORE,PRODUCT_MANUAL_ORDERS_RISK_SCORE,SHORT_REMIT_RISK_SCORE
0,BLC01,2025-09-20,0.842112,0.500000,0.013710,0.014951,0.000000,0.000000,0.145757,NaN,...,0.000000,0.018022,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,BAO01,2025-09-20,0.825651,0.430075,0.000000,0.499210,0.000000,0.000000,0.091817,NaN,...,0.047185,0.047599,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SCN01,2025-09-20,0.857242,0.406463,0.000000,0.210004,0.241084,0.688894,0.414772,NaN,...,0.000000,0.013334,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BJS01,2025-09-20,0.323289,0.413312,0.005499,0.020825,0.000000,0.863735,0.708144,NaN,...,0.000000,0.084018,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SCZ01,2025-09-20,0.821366,0.412756,0.003485,0.400223,0.272825,0.000000,0.833605,NaN,...,0.102690,0.025874,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_sites = df_audit_grade[['Shop Code']].copy()
df_sites.rename(columns={'Shop Code':'SITE_ID'},inplace=True)
df_sites = pd.concat([df_sites,df_kpi_scores[['SITE_ID']]])
df_sites.drop_duplicates(inplace=True)

In [7]:
le = LabelEncoder()
df_sites['SITE_ID_EN'] = le.fit_transform(df_sites[['SITE_ID']])
joblib.dump(le, 'I:\My Drive\Projects_mdw\BayesAudit\Modelssite_label_encoder.pkl')
df_sites.head()

c:\Users\mmadu\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:116: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,SITE_ID,SITE_ID_EN
0,BAA01,0
1,BAB01,1
2,BAC01,2
3,BAD01,3
4,BAE01,4


In [8]:
df_audit_grade.columns

Index(['Shop Code', 'Shop Grade', 'Current Audit Date'], dtype='object')

In [9]:
df_audit_grade['Site_ID'] = le.transform(df_audit_grade[['Shop Code']])
df_audit_grade['Site_ID'] = df_audit_grade['Site_ID'].apply(lambda x:'S' +str(x+1).zfill(3)) 
print(df_audit_grade['Site_ID'].nunique())
print(df_audit_grade.shape)
df_audit_grade[[ 'Site_ID', 'Shop Grade', 'Current Audit Date']].head()

413
(413, 4)


c:\Users\mmadu\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:134: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


,Site_ID,Shop Grade,Current Audit Date
0,S001,B,2025-09-11
1,S002,C-,2025-08-11
2,S003,B,2025-05-14
3,S004,A,2025-07-09
4,S005,B,2025-02-06


In [10]:
df_audit_grade[[ 'Site_ID', 'Shop Grade', 'Current Audit Date']].to_parquet(r'I:\My Drive\Projects_mdw\BayesAudit\Data\audit_grading.parquet',index=False)

In [11]:
# # load Encoder
# le = joblib.load(r'I:\My Drive\Projects_mdw\BayesAudit\Models\site_label_encoder.pkl')

In [12]:
df_kpi_scores['Site_ID'] = le.transform(df_kpi_scores[['SITE_ID']])
df_kpi_scores['Site_ID'] = df_kpi_scores['Site_ID'].apply(lambda x:'S' +str(x+1).zfill(3)) 
print(df_kpi_scores['Site_ID'].nunique())
print(df_kpi_scores.shape)
df_kpi_scores[['SITE_ID','Site_ID']].head()

487
(58013, 25)


c:\Users\mmadu\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:134: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


,SITE_ID,Site_ID
0,BLC01,S254
1,BAO01,S013
2,SCN01,S467
3,BJS01,S231
4,SCZ01,S478


In [13]:
print(df_kpi_scores['Site_ID'].nunique())
print(df_sites['SITE_ID'].nunique())

487
487


In [14]:
df_kpi_scores.drop(columns=['SITE_ID'],axis=1,inplace=True)
df_kpi_scores.head()

,EDATE,INVENTORY_TURNOVER_RTO_RISK_SCORE,PRODUCTRELEASE_COST_RTO_RISK_SCORE,BLACKLIST_CUST_ACTIVE_ACC_RISK_SCORE,PH_NO_CHANGE_CUST_RISK_SCORE,EARL_CLOS_DISCOUNT_RISK_SCORE,HIRE_SALE_ASS_ACC_RISK_SCORE,CASHA_RISK_SCORE,ACC_LESS_ADD_PAY_RISK_SCORE,ISSUE_RTO_RISK_SCORE,...,TOT_HP_ARREARS_RISK_RTO,TOT_CRDT_OUTSTN_AMT_DUE_DAYS_RLN,RETURN_CHEQUE_RISK_SCORE,SPECIAL_DISCOUNT_RISK_SCORE,MULTI_HP_ACC_PER_CUST_RISK_SCORE,INITIAL_PAYMENT_NOT_PAID_RISK_SCORE,SALES_RETURN_RISK_SCORE,PRODUCT_MANUAL_ORDERS_RISK_SCORE,SHORT_REMIT_RISK_SCORE,Site_ID
0,2025-09-20,0.842112,0.500000,0.013710,0.014951,0.000000,0.000000,0.145757,NaN,0.0,...,0.018022,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S254
1,2025-09-20,0.825651,0.430075,0.000000,0.499210,0.000000,0.000000,0.091817,NaN,0.0,...,0.047599,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S013
2,2025-09-20,0.857242,0.406463,0.000000,0.210004,0.241084,0.688894,0.414772,NaN,0.0,...,0.013334,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S467
3,2025-09-20,0.323289,0.413312,0.005499,0.020825,0.000000,0.863735,0.708144,NaN,0.0,...,0.084018,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S231
4,2025-09-20,0.821366,0.412756,0.003485,0.400223,0.272825,0.000000,0.833605,NaN,0.0,...,0.025874,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,S478


In [15]:
df_kpi_scores.isnull().sum()/df_kpi_scores.shape[0]*100

EDATE                                    0.000000
INVENTORY_TURNOVER_RTO_RISK_SCORE        7.546584
PRODUCTRELEASE_COST_RTO_RISK_SCORE      14.627756
BLACKLIST_CUST_ACTIVE_ACC_RISK_SCORE    24.542775
PH_NO_CHANGE_CUST_RISK_SCORE            30.257011
EARL_CLOS_DISCOUNT_RISK_SCORE           35.974695
HIRE_SALE_ASS_ACC_RISK_SCORE            35.974695
CASHA_RISK_SCORE                        55.311740
ACC_LESS_ADD_PAY_RISK_SCORE             61.038043
ISSUE_RTO_RISK_SCORE                    20.993570
cash_coll_vs_banking_rto                 0.000000
UNSALEABLE_INVENTORY_RTO                 7.546584
REVERT_INVENTORY_RTO                     7.546584
TOT_UNPAID_CASH_SALES_RISK              14.627756
TOT_HP_ARREARS_RISK_RTO                 20.285109
TOT_CRDT_OUTSTN_AMT_DUE_DAYS_RLN         0.000000
RETURN_CHEQUE_RISK_SCORE                77.601572
SPECIAL_DISCOUNT_RISK_SCORE             75.439988
MULTI_HP_ACC_PER_CUST_RISK_SCORE        75.439988
INITIAL_PAYMENT_NOT_PAID_RISK_SCORE     80.483685


In [16]:
df_kpi_scores.to_parquet(r'I:\My Drive\Projects_mdw\BayesAudit\Data\kpi_scores.parquet',index=False)